# Functional Analysis Dataset Creation

**Instructor-only notebook** -- generates `functional_analysis_data.csv` and
`functional_analysis_ambiguous.csv` for Part 2 of the Week 8 lab.

## Design Rationale

Part 2 asks students to run Bayesian updating over four candidate functions. Two datasets are
supplied so that students see the procedure under both easy and hard conditions:

- **`functional_analysis_data.csv`** is an unambiguous, attention-maintained case. Rates in the
  attention condition (roughly 8-11 per minute) are far above escape (~0.6-1.5), tangible
  (~1.6-2.5), and play (~0.1-0.6). The posterior resolves within the first cycle. This is the
  reference case: it shows what the machinery does when the data are clean, and it makes the role
  of the control condition visible, since a single elevated attention session on its own is
  equally consistent with an attention function and an automatic function.
- **`functional_analysis_ambiguous.csv`** is a hard case. The true function is escape, but the
  elevation is small (2.6 vs a base rate of 2.0 per minute) and sessions are short (5 min), so
  each session carries little evidence. The posterior takes several cycles to resolve and moves
  non-monotonically along the way. This is where the comparison against visual analysis is
  actually interesting.

Both files carry `count` and `duration_min` alongside `rate_per_min`, because the lab's likelihood
is a Poisson model over counts. Rate alone cannot support it: 3 responses in 1 minute and 30 in 10
minutes give the same rate but very different amounts of evidence.

Conditions rotate in a fixed order in both files: attention, escape, tangible, play.

In [ ]:
import numpy as np
import pandas as pd

CONDITION_ORDER = ["attention", "escape", "tangible", "play"]
N_CYCLES = 10

## Dataset 1: Clear, Attention-Maintained

| Condition  | Mean rate | Notes                            |
|------------|-----------|----------------------------------|
| Attention  | 9.33      | Elevated -- this is the function  |
| Escape     | 1.05      | Low, operant level               |
| Tangible   | 2.05      | Slightly above control           |
| Play       | 0.33      | Control condition, near-zero     |

Rates are hand-specified rather than drawn, so the file is stable across runs and the resulting FA
graph is clean. Sessions are 10 minutes.

In [ ]:
CLEAR_DURATION = 10.0

clear_rates = {
    "attention": [8.4, 9.1, 7.9, 10.2, 8.7, 9.5, 11.0, 8.2, 9.8, 10.5],
    "escape":    [1.2, 0.8, 1.5, 0.6, 1.1, 0.9, 1.3, 0.7, 1.0, 1.4],
    "tangible":  [2.1, 1.7, 2.4, 1.9, 2.0, 1.6, 2.3, 1.8, 2.5, 2.2],
    "play":      [0.3, 0.5, 0.2, 0.4, 0.1, 0.3, 0.6, 0.2, 0.4, 0.3],
}

print("Condition means:")
for cond in CONDITION_ORDER:
    print(f"  {cond:10s}: M = {np.mean(clear_rates[cond]):.2f}, "
          f"SD = {np.std(clear_rates[cond], ddof=1):.2f}")

### Verify the Attention-Maintained Pattern

Every attention session should exceed the highest rate in any other condition in the same cycle.

In [ ]:
for i in range(N_CYCLES):
    attn = clear_rates["attention"][i]
    others_max = max(clear_rates["escape"][i], clear_rates["tangible"][i],
                     clear_rates["play"][i])
    assert attn > others_max, f"Cycle {i}: attention ({attn}) not highest!"

print("All cycles confirmed: attention rate is always the highest condition.")

In [ ]:
rows = []
session = 1
for cycle in range(N_CYCLES):
    for cond in CONDITION_ORDER:
        rate = clear_rates[cond][cycle]
        rows.append({
            "session": session,
            "condition": cond,
            "duration_min": CLEAR_DURATION,
            "count": int(round(rate * CLEAR_DURATION)),
            "rate_per_min": rate,
        })
        session += 1

clear_df = pd.DataFrame(rows)
print(clear_df.groupby("condition")[["count", "rate_per_min"]].mean().round(2))
print(f"\nShape: {clear_df.shape}")
clear_df.head(8)

## Dataset 2: Ambiguous, Escape-Maintained

Counts are drawn from a Poisson distribution: 2.6 per minute in the escape condition and 2.0 per
minute everywhere else, over 5-minute sessions. The separation is deliberately small. A single
session carries a log-likelihood ratio of well under one nat, so the posterior has to accumulate
evidence across cycles rather than resolving on the first session.

The seed is fixed so the file is reproducible. It was chosen for a trajectory that rises steadily,
shows early competition among the wrong hypotheses, and reaches high confidence only after several
cycles.

In [ ]:
AMBIG_DURATION = 5.0
AMBIG_BASE = 2.0        # responses per minute in non-maintaining conditions
AMBIG_ELEVATED = 2.6    # responses per minute in the maintaining condition
AMBIG_FUNCTION = "escape"

rng = np.random.default_rng(1)

rows = []
session = 1
for cycle in range(N_CYCLES):
    for cond in CONDITION_ORDER:
        lam = AMBIG_ELEVATED if cond == AMBIG_FUNCTION else AMBIG_BASE
        count = int(rng.poisson(lam * AMBIG_DURATION))
        rows.append({
            "session": session,
            "condition": cond,
            "duration_min": AMBIG_DURATION,
            "count": count,
            "rate_per_min": round(count / AMBIG_DURATION, 2),
        })
        session += 1

ambig_df = pd.DataFrame(rows)
print(ambig_df.groupby("condition")[["count", "rate_per_min"]].mean().round(2))
print(f"\nShape: {ambig_df.shape}")

### Confirm the Ambiguous Set Is Actually Ambiguous

Two checks. First, the escape condition should have the highest mean but should *not* be highest
in every cycle, otherwise visual analysis would settle it immediately. Second, the per-session
evidence should be small: the mean absolute log-likelihood ratio between the true function and its
closest competitor tells you roughly how many sessions are needed.

In [ ]:
from scipy import stats

wins = sum(
    ambig_df.loc[(ambig_df.session > 4 * c) & (ambig_df.session <= 4 * (c + 1))]
            .set_index("condition")["rate_per_min"].idxmax() == AMBIG_FUNCTION
    for c in range(N_CYCLES)
)
print(f"Cycles in which escape had the highest rate: {wins} of {N_CYCLES}")
assert wins < N_CYCLES, "Ambiguous set is not ambiguous -- escape wins every cycle."

llr = [
    abs(stats.poisson.logpmf(r["count"], AMBIG_ELEVATED * r.duration_min)
        - stats.poisson.logpmf(r["count"], AMBIG_BASE * r.duration_min))
    for _, r in ambig_df.iterrows()
]
print(f"Mean |log-likelihood ratio| per session: {np.mean(llr):.2f} nats")

## Save Both Files

In [ ]:
clear_df.to_csv("functional_analysis_data.csv", index=False)
ambig_df.to_csv("functional_analysis_ambiguous.csv", index=False)
print("Saved functional_analysis_data.csv and functional_analysis_ambiguous.csv")